### **Retrieval**

#### **A typical RAG pipeline**

**RAG Pipeline**

```text
Documents
   ↓
Loading
   ↓
Chunking
   ↓
Embedding
   ↓
Vector Store
   ↓
Retriever
   ↓
LLM
   ↓
Final Answer

#### **What is a retriever?**

##### **Definition**

A retriever takes a user query and returns the most relevant documents or chunks from a knowledge source(VectorDB or VectorStore)

A retriever normally does not generate the final answer itself; it collects the relevant context required to generate the answer.

A retriever is a component that:
- Takes a user query 

- Searches the knowledge source 

- Returns the most relevant documents

A retriever’s job is to find relevant information based on the user’s question.

- User question
- Retriever searches the stored documents
- Returns the most relevant chunks 
- The LLM uses those chunks to generate the answer

**Simple example**

Suppose a vector database contains 1,000 chunks extracted from PDF documents.

The user asks: "What is semantic chunking?"

The retriever does not send all 1,000 chunks to the LLM. It searches the database and returns only the relevant chunks (context/retrieved data/ranked data):

- Chunk 12 → Definition of semantic chunking
- Chunk 48 → Example of semantic chunking
- Chunk 91 → Advantages of semantic chunking

These relevant chunks are then passed to the LLM.

**Simple analogy**

Think of a retriever as a librarian:
- User = Student
- Documents = Library books
- Retriever = Librarian
- LLM = Teacher

The student asks a question. 

The librarian finds the most relevant books or pages and gives them to the teacher. 

The teacher reads those pages and generates the final answer.

**Dummy Indicative code**

retriever = vector_store.as_retriever(
 search_type="mmr",
 search_kwargs={
 "k": 4, # Final number of documents to return
 "fetch_k": 20 # Candidate documents considered by MMR
 }
)

*Retrieve relevant documents*

documents = retriever.invoke("What is a vector database?")

*Display the retrieved content and metadata => Relevant context for the LLM to answer*

for document in documents:
 print(document.page_content)
 print(document.metadata)
 print("-" * 50)


**Parameters in a dummy code**
- k = final number of results retrieved from vectorstore
- filter = metadata restriction
- score_threshold = minimum acceptable relevance score. Referes on the lines of similarity search say for example cosine similarity between -1 to 1(full match) 
- fetch_k = number of candidates fetched before MMR. 
- lambda_mult = balance between relevance and diversity in MMR(maximum marginal retrieval). MMR is advanced type of similarity search which works on cosine similarity itself but in a diff way. 
- search_type → decides which search algorithm runs, out of three available i.e. cosine similarity, dot product, Euclidian distance. 

xyz
- similarity → returns the most similar chunks
- similarity_score_threshold → returns only the chunks that pass the threshold
- mmr → returns relevant and diverse chunks
- search_kwargs → configures the selected search algorithm

##### **1) Similarity Search Methods**

The most commonly supported similarity metrics :

**1) Cosine Similarity** 

Measures the angle/direction similarity between two vectors.
- Higher score = More similar 
- Most commonly used for text embeddings

**2) Euclidean Distance — L2**

Measures the straight-line distance between two vectors.
- Smaller distance = More similar

**3) Dot Product / Inner Product**

Multiplies corresponding vector values and adds them.

- Higher score = More similar
- With normalized vectors, dot product becomes equivalent to cosine similarity.


**Summary**

| Method | Best result |
|---|---|
| Cosine Simialirty | `Highest Score` |
| Euclidean distance | `Lowest Distance` |
| Dot Product | `Highest Score` |

##### **2) Metadata Filtering**

A retriever should not rely only on semantic similarity. It should also support structured constraints using document metadata.
Metadata helps the retriever limit the search to documents that satisfy specific conditions such as department, year, document type, year, source, user role, etc.


**Example Metadata**

metadata = {
 "department": "HR",
 "year": 2026,
 "document_type": "policy",
 "access_role": "manager"
}

*User Query*: Show the HR leave policy for 2026.

Metadata Filter

filter = {
 "department": "HR",
 "year": 2026
}

The retriever will search only the documents whose metadata matches:
department = HR
year = 2026

It will ignore documents from other departments or years, even when their content is semantically similar to the query.

**Common Types of Metadata Filters**
1) *Exact-match filters* : 
Match an exact metadata value.
{"department": "HR"}

2) *Range filters* : 
Match values within a range.
{"year": {"$gte": 2024, "$lte": 2026}}

3) *Boolean filters* : 
Combine multiple conditions using AND, OR, or NOT.
{
 "$and": [
 {"department": "HR"},
 {"year": 2026}
 ]
}
4) *Date filters* : Retrieve documents created or updated within a specific date range.

5) *Department filters* : 
Restrict retrieval to departments such as HR, Finance, Legal, or Engineering.
6) *Document-type filters* : 
Restrict retrieval to policies, reports, invoices, manuals, or contracts.
7) *Source filters* :
Search only selected PDFs, websites, databases, or repositories.
8) *Tenant filters* :
Ensure that users can retrieve documents only from their own organization or tenant.
9) *Role-based access filters* :
Restrict documents according to roles such as employee, manager, HR, or administrator.
10) *Pre-filtering and post-filtering* :
Decide whether metadata restrictions are applied before or after the retrieval operation. 

Metadata Filtering broadly can be categorised into 2 types : 

A) Pre-filtering : Filter first → Search later

B) Post-filtering : Search first → Filter later

**Pre-filtering**

It means applying metadata conditions before running vector or keyword search. 

All stored documents ==> Apply metadata filter ==> Allowed documents only ==> Similarity or keyword search ==> Final results

Example: 

Suppose the vector database contains 10,000 documents:
- HR documents = 1,000
- Finance documents = 3,000
- Engineering documents = 4,000
- Legal documents = 2,000

The user asks:

Show the HR leave policy for 2026.

The filter is:
filter = {
 "department": "HR",
 "year": 2026
}

With pre-filtering:
```
10,000 documents
 ↓
Filter department = HR and year = 2026
 ↓
Only 150 permitted documents remain
 ↓
Similarity search runs on those 150 documents
 ↓
Most relevant HR leave-policy chunks are returned
```


**Code Example**

retriever = vector_store.as_retriever(
 search_type="similarity",
 search_kwargs={
                "k": 4,
                "filter": {
                            "department": "HR",
                            "year": 2026
                            }
                }
                                    )

documents = retriever.invoke("Show the HR leave policy for 2026.")

**Advantages of Pre-filtering**
- Searches a smaller document set
- Reduces irrelevant results
- Improves security
- Supports tenant isolation
- Prevents unauthorized documents from entering the candidate list
- Can improve retrieval speed 

**Post-filtering**

Post-filtering means running retrieval first and applying metadata conditions afterward.

```
All stored documents
 ↓
Similarity or keyword search
 ↓
Top candidate documents
 ↓
Apply metadata filter
 ↓
Final allowed results
```

**Example**

Suppose the retriever first returns the top five semantically similar documents:

Result 1 → Finance leave policy, 2026

Result 2 → HR leave policy, 2025

Result 3 → Legal leave guideline, 2026

Result 4 → HR leave policy, 2026

Result 5 → Engineering leave policy, 2026

Now the filter is applied:

filter = {
 "department": "HR",
 "year": 2026
}

After post-filtering, only one result remains:

Result 4 → HR leave policy, 2026

The retriever originally fetched five documents, but four were removed after retrieval.

**Code Example**
documents = vector_store.similarity_search(
 "Show the HR leave policy for 2026.",
 k=5
)

filtered_documents = [
 document
 for document in documents
 if document.metadata.get("department") == "HR"
 and document.metadata.get("year") == 2026
]

**Limitations of Post-filtering**

- It may return too few final results
- Relevant permitted documents may never enter the initial top-k
- Unauthorized documents may enter the intermediate candidate set
- It is less suitable for strict access control
- A larger initial k may be required

For example:
```
Initial retrieval returns top 5
 ↓
4 results fail the filter
 ↓
Only 1 final result remains
```
Even though more valid HR documents may exist in the database, they may not have appeared in the original top five

### **Retrieval Types**

#### **Sparse Retrieval**

Sparse retrieval searches using exact keywords and term matching.

**Example**

Query: "employee leave policy"

Returns documents containing words such as:employee, leave, policy

Common methods:

- BM25
- TF-IDF
- Keyword Search

#### **Dense Retrieval**

Dense retrieval uses embeddings to understand the semantic meaning of the query.

**Example**

Query: "How many days off can employees take?"

It can retrieve:"Employees are entitled to 20 days of annual leave."

The exact words may be different, but the meaning is similar

#### **Hybrid Retrieval**

Hybrid retrieval combines sparse and dense retrieval.

Keyword/BM25 Search 

    +

Vector Search 

    ↓

`Combined Results`

**Example**:

Query: "HR leave policy 2026"

Sparse retrieval matches exact terms such as:
HR
leave policy
2026

Dense retrieval finds semantically similar content such as:
employee annual vacation guidelines

Hybrid retrieval combines both results for better accuracy.